# Explain Model Predictions with Amazon SageMaker Clarify

There are expanding business needs and legislative regulations that require explainations of _why_ a model mades the decision it did. SageMaker Clarify uses SHAP to explain the contribution that each input feature makes to the final decision.

In [1]:
import boto3
import sagemaker
import pandas as pd
import numpy as np

sess = sagemaker.Session()
bucket = sess.default_bucket()
role = sagemaker.get_execution_role()
region = boto3.Session().region_name

sm = boto3.Session().client(service_name="sagemaker", region_name=region)

sagemaker.config INFO - Not applying SDK defaults from location: /etc/xdg/sagemaker/config.yaml
sagemaker.config INFO - Not applying SDK defaults from location: /home/sagemaker-user/.config/sagemaker/config.yaml


In [2]:
import matplotlib.pyplot as plt

%matplotlib inline
%config InlineBackend.figure_format='retina'

# Test data for explainability

We created test data in JSONLines format to match the model inputs. 

In [3]:
test_data_explainability_path = "./data-clarify/test_data_explainability.jsonl"

In [4]:
!head -n 1 $test_data_explainability_path

{"features":["I have been using Quicken for years now and it does everything that I need it to accomplish for my personal finances.","Digital_Software"]}


### Upload the data

In [5]:
test_data_explainablity_s3_uri = sess.upload_data(
    bucket=bucket, key_prefix="bias/test_data_explainability", path=test_data_explainability_path
)
test_data_explainablity_s3_uri

's3://sagemaker-us-east-1-891377026966/bias/test_data_explainability/test_data_explainability.jsonl'

In [6]:
!aws s3 ls $test_data_explainablity_s3_uri

2024-07-02 19:14:35     190783 test_data_explainability.jsonl


In [7]:
%store test_data_explainablity_s3_uri

Stored 'test_data_explainablity_s3_uri' (str)


# Run Model Explainability Analysis

In [8]:
%store -r training_job_name

In [9]:
try:
    training_job_name
    print("[OK]")
except NameError:
    print("+++++++++++++++++++++++++++++++")
    print("[ERROR] Please run the notebooks in the previous TRAIN section before you continue.")
    print("+++++++++++++++++++++++++++++++")

[OK]


In [10]:
print(training_job_name)

tensorflow-training-2024-07-02-02-08-36-935


## Create Model

In [34]:
import sagemaker

inference_image_uri = sagemaker.image_uris.retrieve(
    framework="tensorflow",
    region=region,
    version="2.3.1",
    py_version="py37",
    instance_type="ml.m5.large",
    image_scope="inference",
)
print(inference_image_uri)

INFO:sagemaker.image_uris:Ignoring unnecessary Python version: py37.


763104351884.dkr.ecr.us-east-1.amazonaws.com/tensorflow-inference:2.3.1-cpu


In [35]:
model_name = sess.create_model_from_job(training_job_name=training_job_name, image_uri=inference_image_uri)
print(model_name)

INFO:sagemaker:Creating model with name: tensorflow-training-2024-07-02-02-08-36-935


tensorflow-training-2024-07-02-02-08-36-935


# SageMakerClarifyProcessor

In [36]:
from sagemaker import clarify

clarify_processor = clarify.SageMakerClarifyProcessor(
    role=role, instance_count=1, instance_type="ml.t3.xlarge", sagemaker_session=sess
)

INFO:sagemaker.image_uris:Defaulting to the only supported framework/algorithm version: 1.0.
INFO:sagemaker.image_uris:Ignoring unnecessary instance type: None.


# Writing DataConfig and ModelConfig
A `DataConfig` object communicates some basic information about data I/O to Clarify. We specify where to find the input dataset, where to store the output, the target column (`label`), the header names, and the dataset type.

Similarly, the `ModelConfig` object communicates information about your trained model and `ModelPredictedLabelConfig` provides information on the format of your predictions.  

**Note**: To avoid additional traffic to your production models, SageMaker Clarify sets up and tears down a dedicated endpoint when processing. `ModelConfig` specifies your preferred instance type and instance count used to run your model on during Clarify's processing.

## DataConfig

In [37]:
explainability_report_prefix = "bias/explainability-report-{}".format(training_job_name)

explainability_output_path = "s3://{}/{}".format(bucket, explainability_report_prefix)

explainability_data_config = clarify.DataConfig(
    s3_data_input_path=test_data_explainablity_s3_uri,
    s3_output_path=explainability_output_path,
    headers=["review_body", "product_category"],
    features="features",
    dataset_type="application/jsonlines",
)

## ModelConfig

In [47]:
model_config = clarify.ModelConfig(
    model_name=model_name,
    instance_type="ml.m5.large",
    instance_count=1,
    content_type="application/jsonlines",
    accept_type="application/jsonlines",
    content_template='{"features":$features}',
)

## SHAPConfig

Here is more information about explainability and SHAP:
* https://docs.aws.amazon.com/sagemaker/latest/dg/clarify-model-explainability.html
* https://docs.aws.amazon.com/sagemaker/latest/dg/clarify-shapley-values.html
* https://papers.nips.cc/paper/2017/file/8a20a8621978632d76c43dfd28b67767-Paper.pdf

In [48]:
shap_config = clarify.SHAPConfig(
    baseline=[{"features": ["ok", "Digital_Software"]}],  # [data.iloc[0].values.tolist()],
    num_samples=5,
    agg_method="mean_abs",
)

# Run Clarify Job

In [49]:
clarify_processor.run_explainability(
    model_config=model_config,
    model_scores="predicted_label",
    data_config=explainability_data_config,
    explainability_config=shap_config,
    wait=False,
    logs=False,
)

INFO:sagemaker.clarify:Analysis Config: {'dataset_type': 'application/jsonlines', 'features': 'features', 'headers': ['review_body', 'product_category'], 'predictor': {'model_name': 'tensorflow-training-2024-07-02-02-08-36-935', 'instance_type': 'ml.m5.large', 'initial_instance_count': 1, 'accept_type': 'application/jsonlines', 'content_type': 'application/jsonlines', 'content_template': '{"features":$features}', 'label': 'predicted_label'}, 'methods': {'report': {'name': 'report', 'title': 'Analysis Report'}, 'shap': {'use_logit': False, 'save_local_shap_values': True, 'baseline': [{'features': ['ok', 'Digital_Software']}], 'num_samples': 5, 'agg_method': 'mean_abs'}}}
INFO:sagemaker:Creating processing-job with name Clarify-Explainability-2024-07-02-20-23-26-501


In [50]:
run_explainability_job_name = clarify_processor.latest_job.job_name
run_explainability_job_name

'Clarify-Explainability-2024-07-02-20-23-26-501'

In [51]:
from IPython.core.display import display, HTML

display(
    HTML(
        '<b>Review <a target="blank" href="https://console.aws.amazon.com/sagemaker/home?region={}#/processing-jobs/{}">Processing Job</a></b>'.format(
            region, run_explainability_job_name
        )
    )
)

/tmp/ipykernel_1012/3963258187.py:1: DeprecationWarning: Importing display from IPython.core.display is deprecated since IPython 7.14, please import from IPython display
  from IPython.core.display import display, HTML


In [52]:
from IPython.core.display import display, HTML

display(
    HTML(
        '<b>Review <a target="blank" href="https://console.aws.amazon.com/cloudwatch/home?region={}#logStream:group=/aws/sagemaker/ProcessingJobs;prefix={};streamFilter=typeLogStreamPrefix">CloudWatch Logs</a> After About 5 Minutes</b>'.format(
            region, run_explainability_job_name
        )
    )
)

/tmp/ipykernel_1012/3786005872.py:1: DeprecationWarning: Importing display from IPython.core.display is deprecated since IPython 7.14, please import from IPython display
  from IPython.core.display import display, HTML


In [53]:
from IPython.core.display import display, HTML

display(
    HTML(
        '<b>Review <a target="blank" href="https://s3.console.aws.amazon.com/s3/buckets/{}?prefix={}/">S3 Output Data</a> After The Processing Job Has Completed</b>'.format(
            bucket, explainability_report_prefix
        )
    )
)

/tmp/ipykernel_1012/3429393802.py:1: DeprecationWarning: Importing display from IPython.core.display is deprecated since IPython 7.14, please import from IPython display
  from IPython.core.display import display, HTML


In [54]:
running_processor = sagemaker.processing.ProcessingJob.from_processing_name(
    processing_job_name=run_explainability_job_name, sagemaker_session=sess
)

processing_job_description = running_processor.describe()

print(processing_job_description)

{'ProcessingInputs': [{'InputName': 'dataset', 'AppManaged': False, 'S3Input': {'S3Uri': 's3://sagemaker-us-east-1-891377026966/bias/test_data_explainability/test_data_explainability.jsonl', 'LocalPath': '/opt/ml/processing/input/data', 'S3DataType': 'S3Prefix', 'S3InputMode': 'File', 'S3DataDistributionType': 'FullyReplicated', 'S3CompressionType': 'None'}}, {'InputName': 'analysis_config', 'AppManaged': False, 'S3Input': {'S3Uri': 's3://sagemaker-us-east-1-891377026966/bias/explainability-report-tensorflow-training-2024-07-02-02-08-36-935/analysis_config.json', 'LocalPath': '/opt/ml/processing/input/config', 'S3DataType': 'S3Prefix', 'S3InputMode': 'File', 'S3DataDistributionType': 'FullyReplicated', 'S3CompressionType': 'None'}}], 'ProcessingOutputConfig': {'Outputs': [{'OutputName': 'analysis_result', 'S3Output': {'S3Uri': 's3://sagemaker-us-east-1-891377026966/bias/explainability-report-tensorflow-training-2024-07-02-02-08-36-935', 'LocalPath': '/opt/ml/processing/output', 'S3Uplo

In [55]:
running_processor.wait(logs=False)

..................................................................................................................................................................................!

# Download Report From S3

In [56]:
!aws s3 ls $explainability_output_path/

                           PRE explanations_shap/
2024-07-02 20:38:23        322 analysis.json
2024-07-02 20:23:27        640 analysis_config.json
2024-07-02 20:38:23     333492 report.html
2024-07-02 20:38:23      60948 report.ipynb
2024-07-02 20:38:23      89733 report.pdf


In [57]:
!aws s3 cp --recursive $explainability_output_path ./explainability_report/

download: s3://sagemaker-us-east-1-891377026966/bias/explainability-report-tensorflow-training-2024-07-02-02-08-36-935/explanations_shap/out.csv to explainability_report/explanations_shap/out.csv
download: s3://sagemaker-us-east-1-891377026966/bias/explainability-report-tensorflow-training-2024-07-02-02-08-36-935/analysis.json to explainability_report/analysis.json
download: s3://sagemaker-us-east-1-891377026966/bias/explainability-report-tensorflow-training-2024-07-02-02-08-36-935/explanations_shap/baseline.csv to explainability_report/explanations_shap/baseline.csv
download: s3://sagemaker-us-east-1-891377026966/bias/explainability-report-tensorflow-training-2024-07-02-02-08-36-935/analysis_config.json to explainability_report/analysis_config.json
download: s3://sagemaker-us-east-1-891377026966/bias/explainability-report-tensorflow-training-2024-07-02-02-08-36-935/report.ipynb to explainability_report/report.ipynb
download: s3://sagemaker-us-east-1-891377026966/bias/explainability-re

In [58]:
from IPython.core.display import display, HTML

display(HTML('<b>Review <a target="blank" href="./explainability_report/report.html">Explainability Report</a></b>'))

/tmp/ipykernel_1012/1351392222.py:1: DeprecationWarning: Importing display from IPython.core.display is deprecated since IPython 7.14, please import from IPython display
  from IPython.core.display import display, HTML


# View the Explainability Report
As with the bias report, you can view the explainability report in Studio under the experiments tab


<img src="img/explainability_detail.gif">

The Model Insights tab contains direct links to the report and model insights.

If you're not a Studio user yet, as with the Bias Report, you can access this report at the following S3 bucket.

# Release Resources

In [59]:
%%html

<p><b>Shutting down your kernel for this notebook to release resources.</b></p>
<button class="sm-command-button" data-commandlinker-command="kernelmenu:shutdown" style="display:none;">Shutdown Kernel</button>
        
<script>
try {
    els = document.getElementsByClassName("sm-command-button");
    els[0].click();
}
catch(err) {
    // NoOp
}    
</script>